# Fine-tuning on Mac M3 / Apple Silicon (Chapter 6)

This notebook accompanies Chapter 6 §6.6 of *Context Engineering with DSPy*. It's the Apple-Silicon path for the fine-tuning section: instead of DSPy's CUDA-only `BootstrapFinetune` (which depends on SGLang), it uses HuggingFace Transformers + `peft` (LoRA) + `trl.SFTTrainer` with PyTorch MPS.

> **Requires Apple Silicon (M1/M2/M3/M4) with MPS available.** Falls back to CPU if MPS is missing, which is *much* slower. Set `TRAINSET_LIMIT`/`TESTSET_LIMIT` (read in the data-load helper, if you add it) or shorten `num_train_epochs` for a smoke test before committing to a full run.


## Setup

In [ ]:
%pip install -r ../requirements.txt -q transformers accelerate trl peft datasets torch

In [ ]:
import dspy
import pandas as pd
import time
import os
import random
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any
from dotenv import load_dotenv

load_dotenv()

# Configuration
LOCAL_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # Small model for Mac
OUTPUT_DIR = Path("finetuned_models")
OUTPUT_DIR.mkdir(exist_ok=True)

NUM_THREADS = 4  # Lower for Mac


@dataclass
class BenchmarkResult:
    optimizer_name: str
    baseline_accuracy: float
    optimized_accuracy: float
    accuracy_uplift: float
    time_seconds: float
    notes: str = ""


def check_mps_availability():
    import torch
    print("=" * 60)
    print("SYSTEM CHECK")
    print("=" * 60)
    print(f"PyTorch version: {torch.__version__}")
    print(f"MPS available: {torch.backends.mps.is_available()}")
    print(f"MPS built: {torch.backends.mps.is_built()}")
    if torch.backends.mps.is_available():
        try:
            x = torch.ones(1, device="mps")
            print(f"MPS test: PASSED")
            return True
        except Exception as e:
            print(f"MPS test: FAILED - {e}")
            return False
    return False


def load_data():
    csv_path = Path('../data/ai_vs_human200.csv')
    df = pd.read_csv(csv_path)
    examples = df.to_dict(orient='records')
    dataset = [dspy.Example(**ex).with_inputs("text") for ex in examples]

    random.seed(42)
    random.shuffle(dataset)
    n = len(dataset)
    train_end = int(n * 0.5)
    val_end = int(n * 0.75)
    trainset = dataset[:train_end]
    valset = dataset[train_end:val_end]
    testset = dataset[val_end:]

    print(f"\nLoaded: {csv_path}")
    print(f"Training set:   {len(trainset)} examples")
    print(f"Validation set: {len(valset)} examples")
    print(f"Test set:       {len(testset)} examples")
    return trainset, valset, testset


class DetectAIText(dspy.Signature):
    """Detects whether text is AI-generated."""
    text: str = dspy.InputField(description="The text to analyze")
    is_ai: bool = dspy.OutputField(description="Whether the text is AI-generated")


class AIDetector(dspy.Module):
    def __init__(self):
        super().__init__()
        self.detect = dspy.ChainOfThought(DetectAIText)

    def forward(self, text: str):
        return self.detect(text=text)


def exact_match(example, response, trace=None):
    return 1 if example.is_ai == response.is_ai else 0


def evaluate_module(module, dataset, metric):
    evaluator = dspy.Evaluate(
        devset=dataset, metric=metric,
        num_threads=NUM_THREADS, display_progress=True, display_table=False,
    )
    result = evaluator(module)
    return result.score / 100


## Fine-tuning runners

In [ ]:
def run_bootstrap_finetune_hf(trainset, valset, testset, baseline_accuracy):
    """
    Run BootstrapFinetune using Hugging Face transformers with MPS.

    This is a manual implementation since DSPy's BootstrapFinetune
    requires SGLang which doesn't support MPS.
    """
    from dspy.teleprompt import BootstrapFewShot
    import torch
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        TrainingArguments,
        Trainer,
        DataCollatorForLanguageModeling,
    )
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model, TaskType

    print("\n" + "=" * 60)
    print("BOOTSTRAP FINETUNE (Hugging Face + MPS)")
    print("=" * 60)

    start_time = time.time()

    # Step 1: Bootstrap successful traces using a teacher model
    print("\nStep 1: Bootstrapping traces with teacher model...")

    # Use OpenAI as teacher to bootstrap demos
    teacher_lm = dspy.LM(
        "openai/gpt-4o-mini",
        api_key=os.getenv("OPENAI_API_KEY"),
        temperature=0.7,
        max_tokens=1000,
    )
    dspy.configure(lm=teacher_lm)

    bootstrap_optimizer = BootstrapFewShot(
        metric=exact_match,
        max_bootstrapped_demos=20,  # Get more demos for training data
        max_labeled_demos=0,
        max_rounds=1,
    )

    teacher_module = AIDetector()
    bootstrapped = bootstrap_optimizer.compile(teacher_module, trainset=trainset)

    # Extract the bootstrapped demos
    demos = []
    for pred_name, predictor in bootstrapped.named_predictors():
        if hasattr(predictor, 'demos') and predictor.demos:
            demos.extend(predictor.demos)

    print(f"Bootstrapped {len(demos)} successful traces")

    if len(demos) < 5:
        print("Not enough successful traces for fine-tuning. Using trainset directly.")
        # Fall back to using trainset examples
        demos = trainset[:20]

    # Step 2: Prepare training data
    print("\nStep 2: Preparing training data...")

    # Format demos into training examples
    training_texts = []
    for demo in demos:
        if hasattr(demo, 'text') and hasattr(demo, 'is_ai'):
            # Create a prompt-completion pair
            prompt = f"Analyze this text and determine if it's AI-generated:\n\nText: {demo.text}\n\nIs AI generated:"
            completion = " Yes" if demo.is_ai else " No"
            training_texts.append({"text": prompt + completion})

    print(f"Created {len(training_texts)} training examples")

    if len(training_texts) == 0:
        return BenchmarkResult(
            optimizer_name="BootstrapFinetune (HF+MPS)",
            baseline_accuracy=baseline_accuracy,
            optimized_accuracy=0,
            accuracy_uplift=0,
            time_seconds=time.time() - start_time,
            notes="FAILED: No training examples could be created from demos",
        )

    # Step 3: Load model and tokenizer
    print(f"\nStep 3: Loading model {LOCAL_MODEL}...")

    device = "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"Using device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL,
        torch_dtype=torch.float32,  # MPS works better with float32
        trust_remote_code=True,
    )

    # Step 4: Configure LoRA for efficient fine-tuning
    print("\nStep 4: Configuring LoRA...")

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],  # Adjust based on model architecture
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Move model to device
    model = model.to(device)

    # Step 5: Tokenize training data
    print("\nStep 5: Tokenizing training data...")

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=512,
            padding="max_length",
        )

    train_dataset = Dataset.from_list(training_texts)
    tokenized_dataset = train_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"],
    )

    # Step 6: Training
    print("\nStep 6: Fine-tuning...")

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / "bootstrap_finetune"),
        num_train_epochs=5,  # More epochs for better learning
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,  # Lower LR for stability
        warmup_steps=20,
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        dataloader_pin_memory=False,  # Required for MPS
        report_to="none",
    )

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator,
    )

    trainer.train()

    # Save the fine-tuned model
    model.save_pretrained(OUTPUT_DIR / "bootstrap_finetune" / "final")
    tokenizer.save_pretrained(OUTPUT_DIR / "bootstrap_finetune" / "final")

    print("\nStep 7: Evaluating fine-tuned model...")

    # Move model to CPU for evaluation to avoid MPS memory issues
    # MPS has a 2GB buffer limit that can cause issues with generation
    print("Moving model to CPU for evaluation (MPS has buffer size limits)...")
    model = model.to("cpu")
    model.eval()

    correct = 0
    total = 0

    for i, example in enumerate(testset):
        prompt = f"Analyze this text and determine if it's AI-generated:\n\nText: {example.text}\n\nIs AI generated:"

        # Truncate long texts to avoid memory issues
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=256,
        )

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response_after_prompt = response[len(prompt):].strip().lower()

        # Check for various indicators of AI-generated text
        ai_indicators = ["yes", "true", "ai", "generated", "artificial"]
        human_indicators = ["no", "false", "human", "not ai", "not generated"]

        ai_score = sum(1 for ind in ai_indicators if ind in response_after_prompt)
        human_score = sum(1 for ind in human_indicators if ind in response_after_prompt)

        # Default to AI if "yes" appears first, or more AI indicators
        predicted_ai = ai_score > human_score

        if predicted_ai == example.is_ai:
            correct += 1
        total += 1

        if (i + 1) % 10 == 0:
            print(f"  Evaluated {i + 1}/{len(testset)} examples...")

    optimized_accuracy = correct / total if total > 0 else 0
    elapsed = time.time() - start_time

    result = BenchmarkResult(
        optimizer_name="BootstrapFinetune (HF+MPS)",
        baseline_accuracy=baseline_accuracy,
        optimized_accuracy=optimized_accuracy,
        accuracy_uplift=optimized_accuracy - baseline_accuracy,
        time_seconds=elapsed,
        notes=f"LoRA fine-tuning on {LOCAL_MODEL} with MPS. {len(training_texts)} training examples.",
    )

    return result


def run_sft_finetune(trainset, valset, testset, baseline_accuracy):
    """
    Run supervised fine-tuning using TRL's SFTTrainer.
    """
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
    from trl import SFTTrainer, SFTConfig
    from datasets import Dataset
    from peft import LoraConfig, TaskType

    print("\n" + "=" * 60)
    print("SUPERVISED FINE-TUNING (TRL SFTTrainer + MPS)")
    print("=" * 60)

    start_time = time.time()

    device = "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"Using device: {device}")

    # Step 1: Prepare training data from trainset
    print("\nStep 1: Preparing training data...")

    training_texts = []
    for example in trainset:
        # Format as instruction-following
        text = f"""### Instruction:
Analyze the following text and determine if it was written by an AI or a human.

### Input:
{example.text}

### Response:
{"This text is AI-generated." if example.is_ai else "This text is human-written."}"""
        training_texts.append({"text": text})

    print(f"Created {len(training_texts)} training examples")

    # Step 2: Load model and tokenizer
    print(f"\nStep 2: Loading model {LOCAL_MODEL}...")

    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

    # Step 3: Configure LoRA
    print("\nStep 3: Configuring LoRA...")

    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    )

    # Step 4: Setup trainer
    print("\nStep 4: Setting up SFTTrainer...")

    train_dataset = Dataset.from_list(training_texts)

    sft_config = SFTConfig(
        output_dir=str(OUTPUT_DIR / "sft_finetune"),
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=10,
        save_steps=50,
        save_total_limit=2,
        max_length=512,  # renamed from max_seq_length in newer TRL
        packing=False,
        report_to="none",
        dataloader_pin_memory=False,  # Required for MPS
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
    )

    # Step 5: Train
    print("\nStep 5: Fine-tuning...")
    trainer.train()

    # Save the model
    trainer.save_model(OUTPUT_DIR / "sft_finetune" / "final")

    # Step 6: Evaluate
    print("\nStep 6: Evaluating fine-tuned model...")

    # Move model to CPU for evaluation to avoid MPS memory issues
    print("Moving model to CPU for evaluation (MPS has buffer size limits)...")
    model = trainer.model.to("cpu")
    model.eval()

    correct = 0
    total = 0

    for i, example in enumerate(testset):
        prompt = f"""### Instruction:
Analyze the following text and determine if it was written by an AI or a human.

### Input:
{example.text}

### Response:
"""

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=256,
        )

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response_after_prompt = response[len(prompt):].strip().lower()

        # Check if prediction matches
        predicted_ai = "ai-generated" in response_after_prompt or "ai generated" in response_after_prompt

        if predicted_ai == example.is_ai:
            correct += 1
        total += 1

        if (i + 1) % 10 == 0:
            print(f"  Evaluated {i + 1}/{len(testset)} examples...")

    optimized_accuracy = correct / total if total > 0 else 0
    elapsed = time.time() - start_time

    result = BenchmarkResult(
        optimizer_name="SFT (TRL + MPS)",
        baseline_accuracy=baseline_accuracy,
        optimized_accuracy=optimized_accuracy,
        accuracy_uplift=optimized_accuracy - baseline_accuracy,
        time_seconds=elapsed,
        notes=f"SFT with LoRA on {LOCAL_MODEL}. {len(training_texts)} training examples.",
    )

    return result


In [ ]:
def print_result(result: BenchmarkResult):
    print("\n" + "=" * 60)
    print(f"RESULT: {result.optimizer_name}")
    print("=" * 60)
    print(f"Baseline Accuracy:   {result.baseline_accuracy*100:.1f}%")
    print(f"Optimized Accuracy:  {result.optimized_accuracy*100:.1f}%")
    print(f"Accuracy Uplift:     {result.accuracy_uplift*100:+.1f}%")
    print(f"Time Taken:          {result.time_seconds:.1f}s")
    if result.notes:
        print(f"Notes:               {result.notes}")
    print("=" * 60)


## Run + results

In [ ]:
# Main: check MPS, load data, configure DSPy, run baseline, then both fine-tuners.
mps_available = check_mps_availability()
if not mps_available:
    print("\nWARNING: MPS not available. Will use CPU (much slower).")

trainset, valset, testset = load_data()

task_lm = dspy.LM(
    "openai/gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=1.0,
    max_tokens=1000,
)
dspy.configure(lm=task_lm)

print("\n" + "=" * 60)
print("BASELINE EVALUATION")
print("=" * 60)
baseline_accuracy = evaluate_module(AIDetector(), testset, exact_match)
print(f"\nBaseline accuracy: {baseline_accuracy*100:.1f}%")

all_results = []

try:
    result = run_bootstrap_finetune_hf(trainset, valset, testset, baseline_accuracy)
    print_result(result)
    all_results.append(result)
except Exception as e:
    print(f"\nBootstrapFinetune failed: {e}")
    import traceback
    traceback.print_exc()

try:
    result = run_sft_finetune(trainset, valset, testset, baseline_accuracy)
    print_result(result)
    all_results.append(result)
except Exception as e:
    print(f"\nSFT Finetune failed: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
for result in all_results:
    print(f"{result.optimizer_name}: {result.optimized_accuracy*100:.1f}% ({result.accuracy_uplift*100:+.1f}%)")
print("\nDone!")
